<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/PCMCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the pcmci family of causal discovery algorithms on the correctedv3 dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
DATASET_PATH = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'

In [3]:
!pip install tigramite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.7/314.7 kB 6.5 MB/s eta 0:00:00


# Objective

- PCMCI+ — single pooled temporal causal discovery ,  Output : Directed lagged graph
- J-PCMCI+ — joint multi-city temporal causal discovery , Output : Joint graph

In [ ]:
import pandas as pd
import numpy as np
import tigramite
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

# 1. Load the dataset
dataset_path = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'
df = pd.read_parquet(dataset_path)

# 2. Define the features requested
requested_features = [
    'workload_causal', 'workload_capped', 'high_load', 'overloaded',
    'pickup_destination_distance', 'batch_size', 'batch_rank_dispatch',
    'batch_rank_capped', 'late_batch', 'extreme_batch', 'hour_sin', 'hour_cos',
    'day_sin', 'day_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve',
    'spatial_congestion_index_daily', 'spatial_congestion_index_rolling7',
    'spatial_congestion_norm', 'courier_local_load', 'WSI', 'precipitation',
    'temperature_2m', 'windspeed_10m', 'is_trajectory_available', 'typecode_cb', 'eta_mins'
]

# Handle wildcard for typecode_grouped_*
grouped_features = [col for col in df.columns if col.startswith('typecode_grouped_')]
requested_features.extend(grouped_features)

# Check which features actually exist in the dataframe to avoid KeyErrors
features = [f for f in requested_features if f in df.columns]
missing = set(requested_features) - set(features)
if missing:
    print(f"Skipping missing columns: {missing}")

# Filter dataframe and handle missing values for PCMCI
selected_df = df[features].dropna().reset_index(drop=True)

# 3. Initialize Tigramite dataframe object
var_names = selected_df.columns.tolist()
data_values = selected_df.values
link_matrix_data = pp.DataFrame(data_values, var_names=var_names)

# 4. Initialize and Run PCMCI+
# Fixed: Changed significance='ait' to 'analytic'
parcorr = ParCorr(significance='analytic')
pcmci = PCMCI(dataframe=link_matrix_data, cond_ind_test=parcorr, verbosity=1)

# Run pcmci_plus
results = pcmci.run_pcmciplus(tau_max=2, pc_alpha=0.05)

# Display summary results
pcmci.print_significant_links(
    p_matrix=results['p_matrix'],
    val_matrix=results['val_matrix'],
    alpha_level=0.05
)

Skipping missing columns: {'spatial_congestion_index_daily', 'spatial_congestion_index_rolling7'}

##
## Step 1: PC1 algorithm for selecting lagged conditions
##

Parameters:
independence test = par_corr
tau_min = 1
tau_max = 2
pc_alpha = [0.05]
max_conds_dim = None
max_combinations = 1



## Resulting lagged parent (super)sets:

    Variable workload_causal has 15 link(s):
        (workload_causal -1): max_pval = 0.00000, |min_val| =  0.190
        (batch_size -1): max_pval = 0.00000, |min_val| =  0.103
        (workload_capped -1): max_pval = 0.00000, |min_val| =  0.063
        (overloaded -1): max_pval = 0.00000, |min_val| =  0.048
        (courier_local_load -2): max_pval = 0.00000, |min_val| =  0.034
        (high_load -1): max_pval = 0.00000, |min_val| =  0.033
        (courier_local_load -1): max_pval = 0.00000, |min_val| =  0.029
        (pickup_destination_distance -1): max_pval = 0.00000, |min_val| =  0.019
        (typecode_cb -1): max_pval = 0.00002, |min_val| =  0.013
    